<a href="https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.dummy import DummyClassifier

# ---- Frozen thresholds (do not tune after model work starts) ----
STALE_DAYS      = 180
MIN_IMPRESSIONS = 500
SLIP_POSITIONS  = 2.0
K               = 50
REVIEW_N        = 20

REASON_CODES = {
    "stale_but_visible": "days_since_update >= 180 and impressions >= 500",
    "position_slipping": "position_now - position_prev >= 2",
    "none": "no condition met",
}
REASON_CODES

{'stale_but_visible': 'days_since_update >= 180 and impressions >= 500',
 'position_slipping': 'position_now - position_prev >= 2',
 'none': 'no condition met'}

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
DATA_PATH = "work/data/pages.csv"        # <- point this at your file
OUT_PATH  = "work/outputs/baseline_action_score.csv"
COLS = dict(id="page_id", age="days_since_update", impr="impressions",
            pos_now="position_now", pos_prev="position_prev", label="label")

def make_demo(n=2000, seed=42):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({
        "page_id": [f"P{i:05d}" for i in range(n)],
        "days_since_update": rng.integers(0, 900, n),
        "impressions": rng.lognormal(6, 1.3, n).astype(int),
        "position_prev": rng.uniform(1, 30, n),
    })
    d["position_now"] = d["position_prev"] + rng.normal(0.5, 3, n)
    d["feature_window_end"] = "2026-06-30"
    d["label_window_start"] = "2026-07-01"
    d["product_flag"] = rng.integers(0, 2, n)      # present in data, NOT used by the rule
    z = (-3.2 + 0.004*d["days_since_update"]
         + 0.25*(d["position_now"]-d["position_prev"])
         + 0.0003*d["impressions"].clip(upper=5000))
    d["label"] = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
    return d

if os.path.exists(DATA_PATH):
    raw = pd.read_csv(DATA_PATH); DATA_SOURCE = "real"
else:
    raw = make_demo(); DATA_SOURCE = "synthetic demo"
print("data source:", DATA_SOURCE, "| rows:", len(raw))

df = raw.copy()
df = df.rename(columns={COLS["id"]:"page_id", COLS["age"]:"days_since_update",
                        COLS["impr"]:"impressions", COLS["pos_now"]:"position_now",
                        COLS["pos_prev"]:"position_prev", COLS["label"]:"label"})

df["position_change"] = df["position_now"] - df["position_prev"]
stale    = (df["days_since_update"] >= STALE_DAYS).astype(int)
visible  = (df["impressions"] >= MIN_IMPRESSIONS).astype(int)
slipping = (df["position_change"] >= SLIP_POSITIONS).astype(int)

df["score"] = stale * visible * slipping * df["impressions"]   # readable on purpose

def codes(a, b, c):
    out = []
    if a and b: out.append("stale_but_visible")
    if c:       out.append("position_slipping")
    return "|".join(out) if out else "none"
df["reason_codes"] = [codes(a, b, c) for a, b, c in zip(stale, visible, slipping)]

ranked = df.sort_values("score", ascending=False, kind="stable").reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
ranked.to_csv(OUT_PATH, index=False)
print("wrote", OUT_PATH, "| flagged (score>0):", int((ranked.score > 0).sum()))

# ---- Evaluation: precision@K next to the base rate ----
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y = ranked["label"].to_numpy()
base_rate = y.mean()
p_at_k = precision_at_k(ranked["score"], y, K)
rng = np.random.default_rng(0)
rand_p = np.mean([y[rng.permutation(len(y))[:K]].mean() for _ in range(500)])
dummy_acc = (DummyClassifier(strategy="most_frequent")
             .fit(ranked[["score"]], y).predict(ranked[["score"]]) == y).mean()

print(f"base rate (labels.mean()):   {base_rate:.3f}")
print(f"random precision@{K}:         {rand_p:.3f}")
print(f"dummy accuracy (majority):   {dummy_acc:.3f}")
print(f"rule precision@{K}:           {p_at_k:.3f}  ({p_at_k/base_rate:.2f}x base rate)")
if (ranked.score > 0).sum() < K:
    print("WARNING: fewer than K rows flagged; precision@K includes zero-score rows.")

data source: synthetic demo | rows: 2000
wrote work/outputs/baseline_action_score.csv | flagged (score>0): 196
base rate (labels.mean()):   0.315
random precision@50:         0.322
dummy accuracy (majority):   0.685
rule precision@50:           0.680  (2.16x base rate)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top = ranked.head(REVIEW_N).copy()

def confidence(r):
    both = "stale_but_visible" in r.reason_codes and "position_slipping" in r.reason_codes
    if both and r.days_since_update >= 365 and r.position_change >= 4:
        return "Higher: all conditions met with clear margin"
    if both:
        return "Medium: all conditions met, but near the thresholds or driven mostly by impressions"
    return "Low: partial signal only"

def wrong_if(r):
    notes = []
    if r.position_change < 3:   notes.append("small slip may be normal rank noise")
    if r.days_since_update < 270: notes.append("only moderately old")
    if r.impressions > ranked.impressions.quantile(0.99): notes.append("score dominated by one very large impressions value (check for a one-off spike)")
    notes.append("page may be evergreen or a utility page that does not need refreshing")
    return "; ".join(notes)

top["action"] = "Review and refresh content; check search intent match"
top["confidence_note"] = top.apply(confidence, axis=1)
top["what_would_make_it_wrong"] = top.apply(wrong_if, axis=1)

review_cols = ["rank","page_id","score","reason_codes","action",
               "confidence_note","what_would_make_it_wrong"]
pd.set_option("display.max_colwidth", 80, "display.width", 200)
top[review_cols]

,rank,page_id,score,reason_codes,action,confidence_note,what_would_make_it_wrong
0,1,P01329,11433,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",score dominated by one very large impressions value (check for a one-off spi...
1,2,P01397,10845,stale_but_visible|position_slipping,Review and refresh content; check search intent match,Higher: all conditions met with clear margin,score dominated by one very large impressions value (check for a one-off spi...
2,3,P00199,9844,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",small slip may be normal rank noise; score dominated by one very large impre...
3,4,P00730,8839,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",score dominated by one very large impressions value (check for a one-off spi...
4,5,P01164,8427,stale_but_visible|position_slipping,Review and refresh content; check search intent match,Higher: all conditions met with clear margin,page may be evergreen or a utility page that does not need refreshing
5,6,P01032,7175,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",page may be evergreen or a utility page that does not need refreshing
6,7,P00982,5438,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",small slip may be normal rank noise; page may be evergreen or a utility page...
7,8,P00738,4787,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",small slip may be normal rank noise; page may be evergreen or a utility page...
8,9,P01944,4695,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",page may be evergreen or a utility page that does not need refreshing
9,10,P00200,4197,stale_but_visible|position_slipping,Review and refresh content; check search intent match,"Medium: all conditions met, but near the thresholds or driven mostly by impr...",page may be evergreen or a utility page that does not need refreshing


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
weak = top[top["label"] == 0].copy()
def why_weak(r):
    why = []
    if r.position_change < 3: why.append("slip barely above threshold")
    if r.days_since_update < 270: why.append("staleness barely above threshold")
    why.append("score rewards impressions, so a big page can outrank a worse page")
    return "; ".join(why)
weak["likely_reason"] = weak.apply(why_weak, axis=1)
print(f"weak picks in top {REVIEW_N}: {len(weak)}")
if len(weak) == 0:
    print("None found. Look harder: check an easier-than-real slice or leakage below.")
weak[["rank","page_id","score","reason_codes","days_since_update","position_change","likely_reason"]]

weak picks in top 20: 5


,rank,page_id,score,reason_codes,days_since_update,position_change,likely_reason
1,2,P01397,10845,stale_but_visible|position_slipping,588,4.164417,"score rewards impressions, so a big page can outrank a worse page"
7,8,P00738,4787,stale_but_visible|position_slipping,664,2.831385,"slip barely above threshold; score rewards impressions, so a big page can ou..."
8,9,P01944,4695,stale_but_visible|position_slipping,342,7.218509,"score rewards impressions, so a big page can outrank a worse page"
9,10,P00200,4197,stale_but_visible|position_slipping,329,5.439479,"score rewards impressions, so a big page can outrank a worse page"
18,19,P00631,3137,stale_but_visible|position_slipping,294,5.510841,"score rewards impressions, so a big page can outrank a worse page"


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.